In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("""
USE CATALOG dbr_dev
""")

spark.sql("""
USE SCHEMA brazilian_ecommerce_bronze
""")

spark.sql("""
SHOW TABLES
""").show()

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/dbr_dev/brazilian_ecommerce_bronze/landing"
    )
)

In [0]:
connection_string = dbutils.secrets.get(
    scope="ecommerce-bronze-scope",
    key="evh-brazilian-ecommerce"
)

eventhub_options = {
    "eventhubs.connectionString": connection_string
}

In [0]:
order_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", DoubleType())
])

In [0]:
raw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**eventhub_options)
    .load()
)

orders_stream = (
    raw_stream
    .select(
        col("body")
        .cast("string")
        .alias("json")
    )
)

In [0]:
orders_parsed = (
    orders_stream
    .select(
        from_json(
            col("json"),
            order_schema
        ).alias("data")
    )
    .select("data.*")
)

display(
    orders_parsed
)